# 08 · Embedding-Based Anomaly Detection

## What this notebook covers
For complex data (text, images, tabular with semantic features) where raw feature space is not informative, embedding-based detection uses pretrained representations. Anomaly scores are computed in the embedding space — typically nearest-neighbour distances to a set of normal embeddings.

## Why embeddings change everything
A pixel-space anomaly detector on images looks at raw pixel values. An embedding-space detector using CLIP or DINOv2 looks at *semantic content*. A bird image doesn't need to be pixel-similar to other birds to be detected as "normal bird" — it needs to be semantically close.

This approach is the foundation of:
- **drift-cam**: production CV semantic drift monitor (Series 9, NB 11 capstone)
- **label-noise**: mislabelled image detection via embedding clustering (backlog)
- Production RAG quality monitoring: detect out-of-distribution queries via embedding distance

## Methods implemented
- **k-NN distance in embedding space**: simplest and most robust; median distance to k nearest neighbours in the normal embedding set
- **Gaussian density in PCA-reduced space**: fit a Gaussian to normal embeddings after PCA; Mahalanobis distance in reduced space

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
np.random.seed(42)

# Simulate embeddings: in production these come from CLIP/DINOv2/sentence-transformers
# Normal: two semantic clusters in 512-d space
dim = 512
n_train = 400
n_test_normal = 200
n_test_anom = 40

# Normal embeddings: sampled from a mixture of two Gaussians
mu_a = np.random.randn(dim) * 0.1
mu_b = np.random.randn(dim) * 0.1 + 0.5
emb_train = np.vstack([
    np.random.randn(200, dim) * 0.1 + mu_a,
    np.random.randn(200, dim) * 0.1 + mu_b,
])
emb_train = normalize(emb_train)

emb_test_normal = normalize(np.vstack([
    np.random.randn(100, dim) * 0.1 + mu_a,
    np.random.randn(100, dim) * 0.1 + mu_b,
]))
# Anomalous: out-of-distribution embeddings
emb_test_anom = normalize(np.random.randn(n_test_anom, dim) * 0.5)

emb_test = np.vstack([emb_test_normal, emb_test_anom])
y_test   = np.array([0]*n_test_normal + [1]*n_test_anom)
print(f"Train: {emb_train.shape}  |  Test: {emb_test.shape} ({y_test.sum()} anomalies)")

In [ ]:
# ── k-NN distance ─────────────────────────────────────────────────────────────
k = 5
nn = NearestNeighbors(n_neighbors=k, metric="cosine")
nn.fit(emb_train)
dists, _ = nn.kneighbors(emb_test)
knn_scores = dists.mean(axis=1)  # mean distance to k nearest normal neighbours

auc_knn = roc_auc_score(y_test, knn_scores)
print(f"k-NN (k={k}) cosine distance AUC: {auc_knn:.4f}")

In [ ]:
# ── Gaussian density in PCA-reduced space ─────────────────────────────────────
n_components = 50
pca = PCA(n_components=n_components, random_state=0)
train_reduced = pca.fit_transform(emb_train)
test_reduced  = pca.transform(emb_test)

cov_model = EmpiricalCovariance()
cov_model.fit(train_reduced)
maha_scores = cov_model.mahalanobis(test_reduced)

auc_maha = roc_auc_score(y_test, maha_scores)
print(f"PCA+Mahalanobis AUC: {auc_maha:.4f}")

In [ ]:
# PCA 2D visualisation
pca2 = PCA(n_components=2, random_state=0)
all_emb = np.vstack([emb_train, emb_test])
all_proj = pca2.fit_transform(all_emb)

train_proj = all_proj[:len(emb_train)]
test_proj  = all_proj[len(emb_train):]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(train_proj[:, 0], train_proj[:, 1], c="lightgrey", s=10, alpha=0.5, label="Train (normal)")
axes[0].scatter(test_proj[y_test==0, 0], test_proj[y_test==0, 1], c="steelblue", s=15, alpha=0.7, label="Test normal")
axes[0].scatter(test_proj[y_test==1, 0], test_proj[y_test==1, 1], c="red", s=60, marker="x", label="Anomaly")
axes[0].set_title("Embedding space (PCA-2D)")
axes[0].legend(fontsize=8)

axes[1].hist(knn_scores[y_test==0], bins=30, alpha=0.7, color="steelblue", label="Normal", density=True)
axes[1].hist(knn_scores[y_test==1], bins=15, alpha=0.7, color="tomato",    label="Anomaly", density=True)
axes[1].set_title(f"k-NN distance distribution (AUC={auc_knn:.3f})")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/08_embedding_anomaly.png", dpi=120)
plt.show()